# 028_polyp_yolo_sam2
## YOLOv12x + SAM2 Zero-Shot Baseline

Baseline pipeline: YOLO detects polyp bounding boxes, SAM2 generates pixel-level segmentation masks from those boxes. No post-processing.

**Self-contained**: This notebook has zero external file dependencies. Any user can run it with their own Google Drive and Colab account, provided they have the polyp dataset and YOLO weights at the paths specified below.


In [ ]:
# Cell 1: Environment Setup
!pip install -q ultralytics
from google.colab import drive
import time
for _ in range(5):
    try:
        drive.mount('/content/drive', force_remount=True)
        break
    except Exception as e:
        print('Mount failed, retrying...', e)
        time.sleep(5)


## Configuration

Edit these paths if your data is in a different location:
- **Dataset**: `/content/drive/MyDrive/data_lake/03_gold/016_polyp_fast_diag_dataset/`
- **YOLO Weights**: `/content/drive/MyDrive/models/trained/025_polyp_yolov12x_aug/train/weights/best.pt`
- **SAM2 Weights**: `sam2_b.pt` (auto-downloaded by ultralytics)
- **Confidence Threshold**: 0.25


In [ ]:
# Cell 3: Evaluate — YOLOv12x + SAM2 Zero-Shot Baseline
import os, glob
from pathlib import Path
import numpy as np
import cv2
from tqdm import tqdm
from ultralytics import YOLO, SAM

def yolo_to_mask(label_path, h, w):
    """Convert YOLO polygon labels to binary mask."""
    mask = np.zeros((h, w), dtype=np.uint8)
    if not os.path.exists(label_path):
        return mask
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 6: continue
            coords = list(map(float, parts[1:]))
            pts = np.array(coords).reshape(-1, 2)
            pts[:, 0] *= w; pts[:, 1] *= h
            cv2.fillPoly(mask, [pts.astype(np.int32)], 1)
    return mask

def extract_mask(sam_results, h, w):
    """Extract binary mask from SAM2 results, handling resize."""
    if sam_results.masks is not None:
        md = sam_results.masks.data.cpu().numpy()
        if md.size > 0:
            if md.ndim == 3: md = md[0]
            if md.shape != (h, w):
                md = cv2.resize(md.astype(np.float32), (w, h), interpolation=cv2.INTER_NEAREST)
            return (md > 0).astype(np.uint8)
    return np.zeros((h, w), dtype=np.uint8)

# --- Config ---
TEST_IMAGES = "/content/drive/MyDrive/data_lake/03_gold/016_polyp_fast_diag_dataset/test/images"
TEST_LABELS = "/content/drive/MyDrive/data_lake/03_gold/016_polyp_fast_diag_dataset/test/labels"
YOLO_WEIGHTS = "/content/drive/MyDrive/models/trained/025_polyp_yolov12x_aug/train/weights/best.pt"
SAM_WEIGHTS = "sam2_b.pt"
CONF = 0.25

# --- Load Models ---
yolo_model = YOLO(YOLO_WEIGHTS)
sam_model = SAM(SAM_WEIGHTS)

test_images = sorted(glob.glob(os.path.join(TEST_IMAGES, '*.*')))
ious, dices = [], []

for img_path in tqdm(test_images, desc='Evaluating'):
    img = cv2.imread(img_path)
    if img is None: continue
    h, w = img.shape[:2]
    label_path = os.path.join(TEST_LABELS, Path(img_path).stem + '.txt')
    gt_mask = yolo_to_mask(label_path, h, w)

    yolo_results = yolo_model(img, conf=CONF, verbose=False)[0]
    boxes = yolo_results.boxes.xyxy.cpu().numpy()
    pred_mask = np.zeros((h, w), dtype=np.uint8)
    if len(boxes) > 0:
        for box in boxes:
            pred_mask = np.logical_or(pred_mask, extract_mask(
                sam_model(img, bboxes=box.tolist(), verbose=False)[0], h, w)).astype(np.uint8)

    intersection = np.logical_and(gt_mask, pred_mask).sum()
    union = np.logical_or(gt_mask, pred_mask).sum()
    ious.append(1.0 if union == 0 else intersection / union)
    denom = gt_mask.sum() + pred_mask.sum()
    dices.append(1.0 if denom == 0 else 2 * intersection / denom)

    # === Metrics ===
    mIoU = np.mean(ious)
    mDice = np.mean(dices)
    print(f"\nResults:")
    print(f"  mIoU:  {mIoU:.4f}")
    print(f"  mDice: {mDice:.4f}")
    print(f"  Images evaluated: {len(ious)}")


In [ ]:
# Cell 4: Cleanup GPU resources
from google.colab import runtime
runtime.unassign()
